# NetworkSet


## 简介



[NetworkSet](../api/networkSet.rst) 对象表示一个无序的网络集合。它
提供遍历和切片集合、按日期时间排序、计算统计量和在图上显示不确定度边界的方法。

## 创建 [NetworkSet](../api/networkSet.rst)

让我们看看 `data/` 文件夹，有一些称为 `ro` 的网络的冗余测量，这是一个*辐射开路*波导。

```
ls data/ro*

-a----       14/02/2021     12:35           8031 ro,1.s1p
-a----       14/02/2021     12:35           8030 ro,2.s1p
-a----       14/02/2021     12:35           8031 ro,3.s1p
-a----       14/02/2021     12:35          46592 ro_spreadsheet.xls
```

文件 `ro,1.s1p` 、`ro,2.s1p`、... 是冗余测量，
我们希望使用 [NetworkSet](../api/networkSet.rst)
类来计算统计量。

[NetworkSet](../api/networkSet.rst) 是从 [Network](../api/network.rst) 的列表或字典创建的。因此首先我们需要将所有 touchstone 文件加载到 `Networks` 中。这可以通过 `rf.io.read_all` 快速完成，参数 `contains` 用于仅加载匹配给定子字符串的文件。

In [ ]:
import skrf as rf

rf.io.read_all(rf.data.pwd, contains='ro')

这可以直接传递给 [NetworkSet](../api/networkSet.rst) 构造函数，

In [ ]:
from skrf.networkSet import NetworkSet

ro_dict = rf.io.read_all(rf.data.pwd, contains='ro')
ro_ns = NetworkSet(ro_dict, name='ro set')
ro_ns

NetworkSet 也可以直接从以下方式构建：
 - 包含 Touchstone 文件的目录：`NetworkSet.from_dir()`，
 - touchstone 文件的 zip 文件：`NetworkSet.from_zip()`，
 - s 参数的字典：`NetworkSet.from_s_dict()`，
 - (G)MDIF (.mdf) 文件：`NetworkSet.from_mdif()`，
 - CITI (.cti) 文件：`NetworkSet.from_citi()`。

## 访问 Network 方法
[NetworkSet](../api/networkSet.rst) 中的 [Network](../api/network.rst) 元素可以像列表元素一样访问，

In [ ]:
ro_ns[0]

大多数 [Network](../api/network.rst) 方法也是 
[NetworkSet](../api/networkSet.rst) 的方法。这些方法在每个 
[Network](../api/network.rst) 元素上单独调用。例如，
绘制每个 Network 的 s 参数的对数幅值。

In [ ]:
%matplotlib inline
import skrf as rf

rf.stylely()

ro_ns.plot_s_db()

## 统计属性


通过访问 NetworkSet 的属性可以计算统计量。要计算集合的复数平均值，访问 `mean_s` 属性

In [ ]:
ro_ns.mean_s

    
统计运算符属性的命名约定是 `NetworkSet.{function}_{parameter}`，其中 `function` 是统计函数的名称，`parameter` 是要操作的 Network 参数。这些方法返回一个 [Network](../api/network.rst) 对象，因此它们可以像 Network 一样保存或绘制。
要绘制复数平均响应的对数幅值

In [ ]:
ro_ns.mean_s.plot_s_db(label='ro')

或者绘制复数 s 参数的标准差，

In [ ]:
ro_ns.std_s.plot_s_re(y_label='Standard Deviations')

使用这些属性可以计算复数网络参数的标量分量的统计量。要计算相位分量的平均值，

In [ ]:
ro_ns.mean_s_deg.plot_s_re()

## 绘制不确定度边界


可以通过以下方法绘制不确定度边界

In [ ]:
ro_ns.plot_uncertainty_bounds_s_db()

In [ ]:
ro_ns.plot_uncertainty_bounds_s_deg()

## 绘制小提琴图

小提琴图用于通过显示每个点的概率密度以及默认的最小值、最大值和平均值来一目了然地了解分布。让我们对频率进行插值以减少混乱。

In [ ]:
ro_ns_interp = ro_ns.interpolate_frequency(rf.Frequency(500, 600, 15, "GHz"))

除时域参数外，大多数常用网络参数都提供小提琴图。要绘制 s 参数的对数幅值和相位

In [ ]:
ro_ns_interp.plot_violin("s_db")

In [ ]:
ro_ns_interp.plot_violin("s_deg")


## 读写

要将 [NetworkSet](../api/networkSet.rst) 的所有 [Network](../api/network.rst) 写出到单独的 touchstone 文件，

In [ ]:
ro_ns.write_touchstone(dir='data/')

对于临时数据存储，[NetworkSet](../api/networkSet.rst) 可以使用函数 `rf.io.read` 和 `rf.io.write` 保存到磁盘和从磁盘读取

    

In [ ]:
rf.io.write('ro set.ns', ro_ns)

In [ ]:
ro_ns = rf.io.read('ro set.ns')
ro_ns

## 导出到 Excel、csv 或 html

[NetworkSet](../api/networkSet.rst) 也可以导出到其他文件类型。输出格式；实部/虚部、幅值/相位是可调的，输出类型也是可调的；csv、excel、html。例如，要将每个网络的幅值/相位导出到 Excel 电子表格给你的老板[s]

In [ ]:
ro_ns.write_spreadsheet('data/ro_spreadsheet.xls', form='db')

有关这方面的更多信息可以在函数 `skrf.io.general.network_2_spreadsheet` 中找到

## 命名参数
如果 `NetworkSet` 的所有 `Network` 对象都有一个 `params` 属性，包含与每个 Network 关联的命名参数和值的字典，则可以使用 `.sel()` 方法选择与命名参数子集对应的 Network。

以下示例说明了此功能。

In [ ]:
# dummy named parameters and values 'a', 'X' and 'c'
import numpy as np

params = [
        {'a':0, 'X':10, 'c':'A'},
        {'a':1, 'X':10, 'c':'A'},
        {'a':2, 'X':10, 'c':'A'},
        {'a':1, 'X':20, 'c':'A'},
        {'a':0, 'X':20, 'c':'A'},
        ]
# create a NetworkSet made of dummy Networks, each define for set of parameters
freq1 = rf.Frequency(75, 110, 101, 'ghz')
rng = np.random.default_rng()
ntwks_params = [rf.Network(frequency=freq1, s=rng.uniform(size=(len(freq1),2,2)),
                               name=f'ntwk_{m}', comment=f'ntwk_{m}', params=params)
                            for (m, params) in enumerate(params) ]
ns = rf.networkSet.NetworkSet(ntwks_params)
print(ns)

选择匹配标量参数的子 NetworkSet 可以如下进行：

In [ ]:
ns.sel({'a': 1})

In [ ]:
ns.sel({'a': 0, 'X': 10})

选择匹配参数范围的子 NetworkSet 也适用：

In [ ]:
ns.sel({'a': 0, 'X': [10,20]})

In [ ]:
ns.sel({'a': [0,1], 'X': [10,20]})

可以使用 `dims` 和 `coords` 属性检索 NetworkSet 的各种命名参数键和值：

In [ ]:
ns.dims

In [ ]:
ns.coords

## 在 NetworkSet 的 Networks 之间插值
可以从 `NetworkSet` 中包含的 Networks 创建插值的新 Networks。如果没有为 NetworkSet 的每个 Network 定义 `params` 属性，可以使用 `interpolate_from_network()` 方法来指定插值参数。

In [ ]:
param_x = [1, 2, 3]  # a parameter associated to each Network
x0 = 1.5  # parameter value to interpolate for
interp_ntwk = ro_ns.interpolate_from_network(param_x, x0)
print(interp_ntwk)

文档的[示例部分](../examples/index.rst#networksets)中给出了一个说明性示例。

当定义了命名参数时，也可以使用命名参数进行插值：

In [ ]:
# Interpolated Network for a=1.2 within X=10 Networks:
ns.interpolate_from_params('a', 1.2, {'X': 10})